# Aula 2 - Introdução aos dados tabulares

In [1]:
%cd "C:/Users/USER/Documents/Projetos/Estudos/Ciencia de dados"

# onde paramos? o log mostra os commits ate aqui
!git log --oneline

# e o estado atual?
!git status

C:\Users\USER\Documents\Projetos\Estudos\Ciencia de dados
8affc6c Adiciona exercicio da aula 2 sobre dados tabulares
e43b165 Documenta a fonte dos dados e ignora a pasta data
5b56495 Adiciona openpyxl para leitura de arquivos Excel
e5fc123 Ajusta comandos do notebook da aula 1 (gitignore e freeze)
7820838 Corrige .gitignore duplicado e requirements.txt sem codigos de cor
a1367c2 Adiciona exercicios e tarefas das unidades 1 e 2
ca949c8 Adicionado o arquivo README.md
fa7582e Adiciona pandas e registra dependencias em requirements.txt
cf36dad Adiciona .gitignore para ambiente, dados e segredos
On branch master
nothing to commit, working tree clean


## Instalando a dependência para ler Excel
O pandas precisa da biblioteca openpyxl para ler arquivos .xlsx.
Instalo no ambiente do projeto (.venv), atualizo o requirements.txt e faço o commit.

In [2]:
# instala dentro do .venv do projeto
!uv pip install --python .venv/Scripts/python.exe openpyxl

# o kernel do notebook usa o Python global, entao ele tambem precisa da biblioteca
%pip install openpyxl

# atualiza a receita com as versoes exatas
!uv pip freeze --color never --python .venv/Scripts/python.exe > requirements.txt
!type requirements.txt

Checked 1 package in 42ms


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


et-xmlfile==2.0.0
numpy==2.5.4
openpyxl==3.1.5
pandas==3.0.6
python-dateutil==2.9.0.post0
six==1.17.0
tzdata==2026.5


In [3]:
!git add requirements.txt
!git commit -m "Adiciona openpyxl para leitura de arquivos Excel"

On branch master
nothing to commit, working tree clean


## Pandas testes iniciais
- Series: uma coluna (valores + índice)
- DataFrame: a tabela inteira (várias Series com o mesmo índice)

In [4]:
import pandas as pd

# DataFrame construido a mao: cinco vendas ficticias
vendas = pd.DataFrame({
    "produto": ["Caneca", "Camiseta", "Caneca", "Adesivo", "Camiseta"],
    "quantidade": [2, 1, 4, 10, 3],
    "preco_unitario": [35.0, 79.9, 35.0, 5.5, 79.9],
    "pais": ["Brasil", "Brasil", "Portugal", "Brasil", "Argentina"],
})

vendas

,produto,quantidade,preco_unitario,pais
0,Caneca,2,35.0,Brasil
1,Camiseta,1,79.9,Brasil
2,Caneca,4,35.0,Portugal
3,Adesivo,10,5.5,Brasil
4,Camiseta,3,79.9,Argentina


Cada linha é uma observação (uma venda) e cada coluna é uma variável.
- `produto` e `pais`: categóricas nominais
- `quantidade`: numérica discreta
- `preco_unitario`: numérica contínua

In [5]:
# 1. selecionar uma coluna: o resultado e uma Series
vendas["preco_unitario"]

0    35.0
1    79.9
2    35.0
3     5.5
4    79.9
Name: preco_unitario, dtype: float64

In [6]:
# 2. criar uma coluna nova a partir das existentes
vendas["faturamento"] = vendas["quantidade"] * vendas["preco_unitario"]
vendas

,produto,quantidade,preco_unitario,pais,faturamento
0,Caneca,2,35.0,Brasil,70.0
1,Camiseta,1,79.9,Brasil,79.9
2,Caneca,4,35.0,Portugal,140.0
3,Adesivo,10,5.5,Brasil,55.0
4,Camiseta,3,79.9,Argentina,239.7


In [7]:
# 3. filtrar linhas por uma condicao (filtro booleano)
# a condicao gera True/False para cada linha e o DataFrame devolve so as True
vendas[vendas["pais"] == "Brasil"]

,produto,quantidade,preco_unitario,pais,faturamento
0,Caneca,2,35.0,Brasil,70.0
1,Camiseta,1,79.9,Brasil,79.9
3,Adesivo,10,5.5,Brasil,55.0


In [8]:
# varias condicoes: use & (e) e | (ou), cada uma entre parenteses
# as palavras and/or do Python nao funcionam com Series
vendas[(vendas["pais"] == "Brasil") & (vendas["faturamento"] > 50)]

,produto,quantidade,preco_unitario,pais,faturamento
0,Caneca,2,35.0,Brasil,70.0
1,Camiseta,1,79.9,Brasil,79.9
3,Adesivo,10,5.5,Brasil,55.0


In [9]:
# describe: estatisticas descritivas das colunas numericas
vendas.describe()

,quantidade,preco_unitario,faturamento
count,5.000000,5.000000,5.000000
mean,4.000000,47.060000,116.920000
std,3.535534,32.307321,75.851282
min,1.000000,5.500000,55.000000
25%,2.000000,35.000000,70.000000
50%,3.000000,35.000000,79.900000
75%,4.000000,79.900000,140.000000
max,10.000000,79.900000,239.700000


## O dataset: Online Retail
Vendas de um varejo online do Reino Unido entre dez/2010 e dez/2011 (541.909 registros, cada linha é um item vendido em uma nota fiscal).
Fonte: UCI Machine Learning Repository (id 352), licença CC BY 4.0. Citação: Chen, D. (2015). Online Retail. UCI.

| Variável | Descrição | Tipo |
|---|---|---|
| InvoiceNo | Número da nota fiscal (começando com C é cancelamento) | categórica nominal |
| StockCode | Código do produto | categórica nominal |
| Description | Nome do produto | categórica nominal |
| Quantity | Quantidade do item na transação | numérica discreta |
| InvoiceDate | Data e hora da transação | temporal |
| UnitPrice | Preço unitário, em libras | numérica contínua |
| CustomerID | Código do cliente | categórica nominal |
| Country | País do cliente | categórica nominal |

### Baixando os dados
Os dados ficam na pasta `data/`, que não deve ir para o GitHub (arquivo grande, dado bruto).
Por isso primeiro coloco `data/` no .gitignore.

In [10]:
from pathlib import Path

# acrescenta data/ no .gitignore so se ainda nao estiver la
gitignore = Path(".gitignore")
texto = gitignore.read_text()
if "data/" not in texto:
    gitignore.write_text(texto + "data/\n")

!type .gitignore

.venv/
__pycache__/
*.env
.ipynb_checkpoints/
data/


In [11]:
pasta_dados = Path("Unidade_01/data")
pasta_dados.mkdir(exist_ok=True)

# baixa o arquivo compactado (cerca de 23 MB)
!curl -L -o Unidade_01/data/online_retail.zip "https://archive.ics.uci.edu/static/public/352/online+retail.zip"

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
100 579.7k   0 579.7k   0      0 440.7k      0           00:01         579.5k
100  7.43M   0  7.43M   0      0  3.21M      0           00:02          3.71M
100 19.72M   0 19.72M   0      0  5.58M      0           00:03          6.13M
100 22.61M   0 22.61M   0      0  6.39M      0           00:03          6.13M
100 22.61M   0 22.61M   0      0  6.39M      0           00:03          6.13M
100 22.61M   0 22.61M   0      0  6.39M      0           00:03          6.13M


In [12]:
import zipfile

# o Windows nao tem unzip no terminal, entao descompacto com o Python
with zipfile.ZipFile(pasta_dados / "online_retail.zip") as zip_dados:
    zip_dados.extractall(pasta_dados)

# renomeia para um nome sem espacos (replace sobrescreve se ja existir)
(pasta_dados / "Online Retail.xlsx").replace(pasta_dados / "online_retail.xlsx")

!dir Unidade_01\data

 O volume na unidade C é SSD
 O Número de Série do Volume é 42EA-C803

 Pasta de C:\Users\USER\Documents\Projetos\Estudos\Ciencia de dados\Unidade_01\data

12/10/2026  18:55    <DIR>          .
12/10/2026  17:07    <DIR>          ..
12/10/2026  18:55        23.715.344 online_retail.xlsx
12/10/2026  18:55        23.715.478 online_retail.zip
               2 arquivo(s)     47.430.822 bytes
               2 pasta(s)   140.381.925.376 bytes disponíveis


In [13]:
# os arquivos novos aparecem no git status? (nao deveriam, data/ esta no .gitignore)
!git status

On branch master
nothing to commit, working tree clean


Os dados não vão para o repositório, então o README precisa ensinar como obtê-los.

In [14]:
# documenta a fonte dos dados no README (so se ainda nao estiver la)
readme = Path("README.md")
texto = readme.read_text(encoding="utf-8")
if "## Dados" not in texto:
    secao = (
        "\n## Dados\n\n"
        "Dataset Online Retail (UCI Machine Learning Repository, id 352).\n"
        "Licenca CC BY 4.0. Citacao: Chen, D. (2015). Online Retail. UCI.\n"
        "Download: https://archive.ics.uci.edu/dataset/352/online+retail\n"
        "Apos baixar, descompacte em Unidade_01/data/ e renomeie para online_retail.xlsx\n"
    )
    readme.write_text(texto + secao, encoding="utf-8")

!type README.md

# Meu Repositório 

Exercícios e projetos organizados por unidade.

- Unidade_01: Aula_01, Aula_02, Tarefa_Unidade_1
- Unidade_02: Aula_03, Tarefa_Unidade_2
- Unidade_03: Aula_04, Tarefa_Unidade_3

## Dados

Dataset Online Retail (UCI Machine Learning Repository, id 352).
Licenca CC BY 4.0. Citacao: Chen, D. (2015). Online Retail. UCI.
Download: https://archive.ics.uci.edu/dataset/352/online+retail
Apos baixar, descompacte em Unidade_01/data/ e renomeie para online_retail.xlsx


In [15]:
!git add README.md .gitignore
!git commit -m "Documenta a fonte dos dados e ignora a pasta data"

On branch master
nothing to commit, working tree clean


## Inspeção inicial
Leitura do Excel. 

In [16]:
# le o Excel (sao 541 mil linhas, demora um pouco)
caminho = pasta_dados / "online_retail.xlsx"

retail = pd.read_excel(caminho)

### Passo 1: olhar os dados

In [17]:
# primeiras linhas
retail.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [18]:
# ultimas linhas (fim de arquivo pode esconder totais e lixo)
retail.tail()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
541908,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


### Passo 2: tamanho e tipos

In [19]:
# quantas linhas e colunas?
retail.shape

(541909, 8)

In [20]:
# tipos, valores nao nulos e memoria
retail.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


Olhando o `info()`: `CustomerID` foi lido como float64. Isso acontece porque a coluna tem valores ausentes
(NaN é um número decimal e o tipo inteiro do pandas não comporta NaN, então tudo vira float).

### Passo 3: valores ausentes

In [21]:
# quantos ausentes por coluna?
# isna() marca True onde falta valor e sum() soma os True de cada coluna
retail.isna().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

### Passo 4: resumo das variáveis numéricas

In [22]:
# resumo das colunas numericas (olhar minimo e maximo de Quantity e UnitPrice)
retail.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


### Passo 5: contagem das categorias

In [23]:
# as 10 maiores origens das vendas
retail["Country"].value_counts().head(10)

Country
United Kingdom    495478
Germany             9495
France              8557
EIRE                8196
Spain               2533
Netherlands         2371
Belgium             2069
Switzerland         2002
Portugal            1519
Australia           1259
Name: count, dtype: int64

In [24]:
# quantos paises distintos?
retail["Country"].nunique()

38

## Seleção e filtragem
Mesmas ferramentas da tabela pequena, agora no dataset real.

In [25]:
# vendas para a Alemanha
vendas_alemanha = retail[retail["Country"] == "Germany"]
vendas_alemanha.shape

(9495, 8)

In [26]:
# transacoes com preco unitario acima de 100 libras fora do Reino Unido
# filtra as linhas e escolhe so as colunas relevantes (lista de nomes entre colchetes)
caras_fora_uk = retail[(retail["UnitPrice"] > 100) & (retail["Country"] != "United Kingdom")]
caras_fora_uk[["InvoiceNo", "Description", "UnitPrice", "Country"]].head()

,InvoiceNo,Description,UnitPrice,Country
39777,539688,CARRIAGE,150.0,France
45622,C540271,Manual,1126.0,Spain
54382,540946,VINTAGE BLUE KITCHEN CABINET,125.0,Cyprus
54395,540946,VINTAGE RED KITCHEN CABINET,125.0,Cyprus
70795,542102,VINTAGE RED KITCHEN CABINET,125.0,Singapore


In [27]:
# loc seleciona por rotulo/condicao, iloc por posicao
print(retail.loc[retail["Country"] == "France", ["InvoiceNo", "UnitPrice"]].head())
print(retail.iloc[0:5, 0:3])

   InvoiceNo  UnitPrice
26    536370       3.75
27    536370       3.75
28    536370       3.75
29    536370       0.85
30    536370       0.65
  InvoiceNo StockCode                          Description
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER
1    536365     71053                  WHITE METAL LANTERN
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.


### Perguntas de negócio
1. Quantas transações registram o país como Brazil?
2. Quantas linhas têm Quantity negativa? Como elas são?
3. Existe linha com UnitPrice igual a zero e CustomerID ausente ao mesmo tempo?

In [28]:
# 1. transacoes do Brasil
brasil = retail[retail["Country"] == "Brazil"]
print(len(brasil), "transacoes do Brazil")

32 transacoes do Brazil


**Interpretação:** só 32 transações têm o país como Brazil, cerca de 0,006% das 541.909 linhas. O Brasil quase não aparece nessa base, que é de um varejo do Reino Unido.

In [29]:
# 2. quantidade negativa
negativas = retail[retail["Quantity"] < 0]
print(len(negativas), "linhas com quantidade negativa")
negativas.head()

10624 linhas com quantidade negativa


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom


**Interpretação:** são 10.624 linhas com quantidade negativa (cerca de 2% da base). Pelas primeiras linhas, muitas vêm de faturas que começam com C e de itens como "Discount", ou seja, são saídas e não vendas normais.

In [30]:
# 3. preco zero E cliente ausente ao mesmo tempo
zero_sem_cliente = retail[(retail["UnitPrice"] == 0) & (retail["CustomerID"].isna())]
print(len(zero_sem_cliente), "linhas com preco zero e sem CustomerID")

2475 linhas com preco zero e sem CustomerID


**Interpretação:** sim, 2.475 linhas têm preço zero e não têm CustomerID. Sem preço e sem cliente, parecem brindes, ajustes internos ou erros de registro, e não vendas a clientes.

## Diagnóstico de qualidade
Aqui só diagnostico os problemas, sem tratar nada. A decisão do que fazer com cada um fica para a próxima aula.

### Suspeita 1: quantidades negativas

In [31]:
# as notas das quantidades negativas comecam com C (cancelamento)?
notas_negativas = retail.loc[retail["Quantity"] < 0, "InvoiceNo"].astype(str)
print("Comecam com C:", notas_negativas.str.startswith("C").sum())
print("Total de negativas:", len(notas_negativas))

Comecam com C: 9288
Total de negativas: 10624


### Suspeita 2: preços zerados

In [32]:
# filtra so as linhas com preco zero e mostra algumas colunas para entender o que sao
preco_zero = retail[retail["UnitPrice"] == 0]
print(len(preco_zero), "linhas com preco zero")
preco_zero[["InvoiceNo", "Description", "Quantity", "CustomerID"]].head()

2515 linhas com preco zero


,InvoiceNo,Description,Quantity,CustomerID
622,536414,NaN,56,NaN
1970,536545,NaN,1,NaN
1971,536546,NaN,1,NaN
1972,536547,NaN,1,NaN
1987,536549,NaN,1,NaN


### Suspeita 3: registros duplicados

In [33]:
# linhas identicas em todas as colunas
print(retail.duplicated().sum(), "linhas duplicadas")

5268 linhas duplicadas


## Ficha de reconhecimento do dataset

- **Tamanho:** 541.909 linhas e 8 colunas, com vendas de dezembro de 2010 a dezembro de 2011.
- **Tipos das variáveis:**
  - `InvoiceNo`, `StockCode`, `Description` e `Country`: categóricas nominais (texto).
  - `Quantity`: numérica discreta (`int64`).
  - `UnitPrice`: numérica contínua (`float64`).
  - `InvoiceDate`: temporal (`datetime64`).
  - `CustomerID`: identificador do cliente, categórica nominal, mas lida como `float64` por causa dos ausentes.
- **Ausentes:** `CustomerID` com 135.080 (24,9% das linhas) e `Description` com 1.454 (0,3%). As demais colunas estão completas.
- **Três problemas de qualidade:**
  1. 10.624 linhas com quantidade negativa, das quais 9.288 são de faturas que começam com C (cancelamentos).
  2. 2.515 linhas com preço unitário zero, e 2.475 delas sem `CustomerID`.
  3. 5.268 linhas duplicadas (idênticas em todas as colunas).

## Pergunta própria

Qual é o produto que mais aparece nas vendas da Alemanha?

In [ ]:
# produtos que mais aparecem nas vendas da Alemanha (so filtro e contagem)
alemanha = retail[retail["Country"] == "Germany"]
print(alemanha["Description"].value_counts().head(5))

Description
POSTAGE                               383
ROUND SNACK BOXES SET OF4 WOODLAND    120
REGENCY CAKESTAND 3 TIER               81
ROUND SNACK BOXES SET OF 4 FRUITS      78
PLASTERS IN TIN WOODLAND ANIMALS       67
Name: count, dtype: int64


**Interpretação:** o item mais frequente na Alemanha é POSTAGE (383 linhas), que é frete e não um produto. Entre os produtos de verdade, o mais comum é ROUND SNACK BOXES SET OF4 WOODLAND (120 linhas). Vale lembrar disso nas próximas aulas, para não tratar frete como produto.

In [35]:
# salva o notebook com as saidas e registra no historico
!git add Unidade_01/aula_pratica_02/exercicio_aula02.ipynb
!git commit -m "Salva as saídas do exercício da aula 2"
!git log --oneline

fatal: pathspec 'Unidade_01/aula_pratica_02/exercicio_aula02.ipynb' did not match any files


On branch master
nothing to commit, working tree clean
8affc6c Adiciona exercicio da aula 2 sobre dados tabulares
e43b165 Documenta a fonte dos dados e ignora a pasta data
5b56495 Adiciona openpyxl para leitura de arquivos Excel
e5fc123 Ajusta comandos do notebook da aula 1 (gitignore e freeze)
7820838 Corrige .gitignore duplicado e requirements.txt sem codigos de cor
a1367c2 Adiciona exercicios e tarefas das unidades 1 e 2
ca949c8 Adicionado o arquivo README.md
fa7582e Adiciona pandas e registra dependencias em requirements.txt
cf36dad Adiciona .gitignore para ambiente, dados e segredos
